In [ ]:
import os
%pwd

In [ ]:
import os

# Change directory to the src folder
os.chdir(r"e:\textsummarizer\src")

# Verify the current working directory
print(f"Current Directory: {os.getcwd()}")

In [ ]:
%pwd

In [ ]:
# %%
import os
%pwd  # Check initial directory

# %%
os.chdir("../") # Move up one directory level

# %%
%pwd # Verify new directory

# %%
from dataclasses import dataclass
from pathlib import Path

# Define a structure to hold model training configuration parameters
@dataclass
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path # Checkpoint path/name for model and tokenizer
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float # Note: HuggingFace TrainingArguments usually expects int for save_steps
    gradient_accumulation_steps: int

# %%
# Import project-specific constants and utility functions
from src.textSummarizer.constants import * # Assuming CONFIG_FILE_PATH, PARAMS_FILE_PATH are defined here
from src.textSummarizer.utils.common import read_yaml, create_directories # Assuming these functions exist


In [ ]:
import os

# Change directory to the location of config.yaml
os.chdir(r"e:\textsummarizer")

# Verify the current working directory
print(f"Current Directory: {os.getcwd()}")

In [ ]:
# %%
# Class to manage reading configuration from YAML files
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        """
        Initializes the ConfigurationManager by reading config and params YAML files.

        Args:
            config_filepath (str/Path): Path to the main configuration YAML file.
            params_filepath (str/Path): Path to the parameters YAML file.
        """
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        # Ensure the root artifact directory exists
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        """
        Extracts model training specific configuration and parameters.

        Returns:
            ModelTrainerConfig: A dataclass object containing all necessary
                                parameters for the ModelTrainer.
        """
        config = self.config.model_trainer
        params = self.params.TrainingArguments # Assumes structure in YAML files

        # Ensure the specific root directory for this trainer exists
        create_directories([config.root_dir])

        # Create and populate the ModelTrainerConfig object
        # Potential issue: Ensure types from YAML match expected types in ModelTrainerConfig
        # Potential issue: eval_steps assigned from evaluation_strategy - likely incorrect
        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir), # Ensure Path object
            data_path=Path(config.data_path), # Ensure Path object
            model_ckpt=config.model_ckpt, # Should be string like "google/pegasus-cnn_dailymail"
            num_train_epochs = int(params.num_train_epochs),
            warmup_steps = int(params.warmup_steps),
            per_device_train_batch_size = int(params.per_device_train_batch_size),
            weight_decay = float(params.weight_decay),
            logging_steps = int(params.logging_steps),
            eval_strategy = str(params.eval_strategy),
            # ---> Possible Bug: Should likely be params.eval_steps <---
            eval_steps = int(params.eval_steps), # Should be int based on TrainingArguments
            # ---> Possible Bug: save_steps is float in dataclass, but int in TrainingArguments <---
            # --->              Usually depends on eval_strategy ('steps' requires int) <---
            save_steps = float(params.save_steps), # Convert carefully based on intended use
            gradient_accumulation_steps = int(params.gradient_accumulation_steps)
        )
        return model_trainer_config

In [ ]:
# %%
# Import necessary components from Hugging Face and PyTorch libraries
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk # To load a pre-processed dataset

In [ ]:

# Class to handle the actual model training process
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        """
        Initializes the ModelTrainer with its configuration.

        Args:
            config (ModelTrainerConfig): Configuration object with paths and hyperparameters.
        """
        self.config = config

    def train(self):
        """
        Executes the model training pipeline.
        """
        # Determine device (use GPU if available)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")

        # Load tokenizer and model from the specified checkpoint
        # Ensure model_ckpt is treated as a string identifier
        print(f"Loading tokenizer from: {self.config.model_ckpt}")
        tokenizer = AutoTokenizer.from_pretrained(str(self.config.model_ckpt))

        print(f"Loading model from: {self.config.model_ckpt}")
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(str(self.config.model_ckpt)).to(device)

        # Initialize data collator for sequence-to-sequence tasks
        print("Initializing Data Collator...")
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        # Load the dataset (previously processed and saved to disk)
        print(f"Loading dataset from disk: {self.config.data_path}")
        dataset_samsum_pt = load_from_disk(str(self.config.data_path)) # Ensure data_path is string
        print(f"Dataset loaded. Original columns: {dataset_samsum_pt['test'].column_names}")

        # -------- START: ADDED TOKENIZATION STEP --------
        print("Starting dataset tokenization...")

        # Define the preprocessing function
        def preprocess_function(examples):
            # Prepare inputs (source text - assuming 'dialogue' column)
            inputs = examples['dialogue']
            # Adjust max_length as needed for your model/data
            model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

            # Prepare targets (summary text - assuming 'summary' column)
            targets = examples['summary']
             # Adjust max_length as needed for your model/data
            with tokenizer.as_target_tokenizer():
                labels = tokenizer(targets, max_length=128, truncation=True)

            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        # Apply the preprocessing function to the whole dataset
        # Use batched=True for efficiency
        tokenized_datasets = dataset_samsum_pt.map(
            preprocess_function,
            batched=True,
            # Optionally remove original columns if not needed later
            # remove_columns=dataset_samsum_pt["train"].column_names
        )
        print("Dataset tokenization complete.")
        print(f"Columns after tokenization: {tokenized_datasets['test'].column_names}") # Verify new columns

        # -------- END: ADDED TOKENIZATION STEP --------


        # --- NOTE: TrainingArguments are still hardcoded as requested ---
        # --- Consider fixing this later to use self.config values ---
        trainer_args = TrainingArguments(
            output_dir=str(self.config.root_dir), # Uses config for output directory
            num_train_epochs=1,                   # Hardcoded
            warmup_steps=500,                     # Hardcoded
            per_device_train_batch_size=1,        # Hardcoded
            per_device_eval_batch_size=1,         # Hardcoded
            weight_decay=0.01,                    # Hardcoded
            logging_steps=10,                     # Hardcoded
            eval_strategy='steps',                # Hardcoded
            eval_steps=500,                       # Hardcoded
            save_steps=1e6,                       # Hardcoded (float might be issue)
            gradient_accumulation_steps=16,       # Hardcoded
            remove_unused_columns=True          # Hardcoded
        )

        # Initialize the Trainer using the TOKENIZED dataset
        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            # Using 'test' for training and 'validation' for eval - check if this is intended
            # USE THE TOKENIZED DATASET
            train_dataset=tokenized_datasets["test"],
            eval_dataset=tokenized_datasets["validation"]
        )

        # Start training
        print("Starting model training...")
        trainer.train()
        print("Training finished.")

        # Save the fine-tuned model and tokenizer
        model_save_path = os.path.join(str(self.config.root_dir), "pegasus-samsum-model")
        tokenizer_save_path = os.path.join(str(self.config.root_dir), "tokenizer")

        print(f"Saving model to {model_save_path}")
        model_pegasus.save_pretrained(model_save_path)

        print(f"Saving tokenizer to {tokenizer_save_path}")
        tokenizer.save_pretrained(tokenizer_save_path)
        print("Model and tokenizer saved.")

# %%

In [ ]:
# %%
# Main execution block
try:
    print("Initializing Configuration Manager...")
    config_manager = ConfigurationManager()
    model_trainer_config = config_manager.get_model_trainer_config()
    print("Configuration loaded successfully.")
    print(f"Model Trainer Config: {model_trainer_config}") # Log the config being used

    print("Initializing Model Trainer...")
    model_trainer = ModelTrainer(config=model_trainer_config)
    print("Model Trainer initialized.")

    # Run the training process
    model_trainer.train()

except FileNotFoundError as e:
    print(f"\nERROR: Configuration file not found.")
    print(f"Details: {e}")
    print(f"Please ensure '{e.filename}' exists relative to the current working directory:")
    print(f"Current Directory: {os.getcwd()}")
except KeyError as e:
    print(f"\nERROR: Missing key in configuration files.")
    print(f"Details: Could not find key {e} in config.yaml or params.yaml.")
    print("Please check the structure of your YAML files.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
    # You might want more detailed logging or re-raising depending on context
    # import traceback
    # traceback.print_exc() # Uncomment for full traceback

print("\nScript finished.")

# %%